# Step 4 · Evaluate

Run a model on validation questions it never trained on and score every answer.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import engines
MODEL = latest("sft") or BASE
DS = latest_ds("sft_val")
rows = data.load(DS)
rid = registry.rewards()[0]["id"]; rules = registry.reward(rid)["rules"]
answers = engines.get(registry.model(MODEL)["base_model"]).generate(MODEL, [r["prompt"] for r in rows], max_new_tokens=96)

In [ ]:
results = []
for r, a in zip(rows, answers):
    s = rewards.score(a, rules, r)
    results.append({"id": r["id"], "question": r["prompt"], "answer": a, "approved": r.get("completion", ""),
                    "reward": s["total"], "breakdown": s["breakdown"], "review": ""})
mean = sum(x["reward"] for x in results) / len(results)
EVAL = registry.add_eval(MODEL, DS, rid, {"max_new_tokens": 96, "temperature": 0, "source": "notebook"}, results, mean)
print(f"evaluation #{EVAL}: mean rule score {mean:.2f}")
pd.DataFrame(results)[["question", "answer", "approved", "reward"]]